In [ ]:
# parameters
# BINDER_FAST: set N=4, n_eps=6, n_delta=5 for fast cloud execution
N = 8                 # Hilbert space truncation
omega0_GHz = 5.0      # bare 0->1 frequency (GHz)
K_GHz = 0.2           # Kerr nonlinearity (GHz)
kappa_MHz = 1.0       # peak single-photon loss rate (MHz)
Gamma_MHz = 100.0     # bath FWHM (MHz)
nbar = 0.02           # thermal occupation
n_eps = 12            # number of points in epsilon dimension of 2D map
n_delta = 10          # number of points in detuning dimension of 2D map

In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import warnings
%matplotlib widget

from bosonic_gates.driven_kerr import (
    DrivenKerrConfig,
    make_operators,
    J,
    run_lindblad,
    run_full_floquet_markov,
    effective_loss_rate,
    compute_floquet_modes,
    assemble_rates_with_dephasing,
)
from bosonic_gates.driven_kerr.floquet_markov import effective_decay_rate_from_R

TWO_PI = 2 * np.pi

## Module 6c: Validity Map of Master Equation Methods

**Learning objectives:**
- Understand why Method A (Lindblad) and Method C (Floquet–Markov) disagree at strong drive
- Extract effective decay rates from both methods and compute their fractional discrepancy
- Build a 2D map of method agreement over $(\varepsilon, \Delta)$ parameter space
- Identify the 10 % validity boundary and overlay the analytic prediction
- Know when to use Method A vs Method C for driven systems

---

**Sections:**
[1 Why Methods Disagree](#sec1) ·
[2 Rate Extraction](#sec2) ·
[3 2D Validity Map](#sec3) ·
[4 Physical Interpretation](#sec4) ·
[5 Analytic Boundary](#sec5) ·
[6 Exercises](#sec6)

<a id="sec1"></a>
## 1  Why Methods A and C Disagree

**Recall from Module 3c:**

- **Method A (Lindblad):** The jump operators are fixed using $J(\omega_0)$ — the
  bath spectral density evaluated at the *bare* transition frequency, regardless of
  the drive amplitude. This is exact only when the bath is infinitely flat (white bath).

- **Method C (Floquet–Markov):** The decay rates are computed using the dressed
  Floquet quasi-energy differences $\Delta_{mn}$ and Floquet-basis coupling amplitudes.
  As $\varepsilon$ increases, these dressed frequencies shift, sampling $J$ at
  different points.

**Physical mechanism of the discrepancy:**

A drive of amplitude $\varepsilon$ shifts the dominant decay sideband frequency by
approximately $\delta\omega_{\rm sb} \approx \varepsilon^2 / \Gamma$ relative to $\omega_0$.
Method A ignores this shift and always uses $J(\omega_0)$. Method C tracks it.

The methods agree when $J(\omega_0 + \delta\omega_{\rm sb}) \approx J(\omega_0)$,
i.e., when the sideband stays within the flat part of the Lorentzian:

$$\delta\omega_{\rm sb} \ll \Gamma/2
\quad\Longleftrightarrow\quad
\varepsilon^2 / \Gamma \ll \Gamma/2
\quad\Longleftrightarrow\quad
\varepsilon \ll \Gamma/\sqrt{2}.$$

**Detuning also matters:** A large drive detuning $\Delta = \omega_d - \omega_0$ shifts
the dressed quasi-energies directly. Even at small $\varepsilon$, large $|\Delta|$
can move the sideband outside the bath Lorentzian, causing disagreement.

**Analytic validity boundary:**
Combining both effects, the approximate boundary for 10 % agreement is:

$$\varepsilon \lesssim \Gamma \sqrt{\Gamma/\kappa},$$

which for our parameters ($\kappa/2\pi = 1$ MHz, $\Gamma/2\pi = 100$ MHz) gives
$\varepsilon/2\pi \lesssim 100 \times \sqrt{100} \approx 1000$ MHz — much larger than
the drive range we sweep. The 10 % boundary is set by the detuning-dependent
sideband shift rather than the simple $\varepsilon^2/\Gamma$ formula alone.

*Lab note: the 2D validity map computed in Section 3 is the definitive reference
for deciding which method to use. Check that your operating point lies in the green
region before trusting Lindblad results.*

In [ ]:
# Build the base configuration
cfg_base = DrivenKerrConfig(
    N=N,
    omega0=TWO_PI * omega0_GHz,
    K=TWO_PI * K_GHz,
    omega_d=TWO_PI * (omega0_GHz + 0.005),  # 5 MHz blue-detuned (breaks Floquet degeneracy)
    omega_f=TWO_PI * omega0_GHz,             # bath center at omega0
    kappa=TWO_PI * kappa_MHz * 1e-3,
    Gamma=TWO_PI * Gamma_MHz * 1e-3,
    nbar=nbar,
    gamma_phi=TWO_PI * 0.05e-3,
    k_max=5,
    n_t=512,
)

print("Base configuration:")
print(f"  ω₀/2π = {cfg_base.omega0/TWO_PI:.3f} GHz")
print(f"  K/2π  = {cfg_base.K/TWO_PI*1e3:.0f} MHz")
print(f"  κ/2π  = {cfg_base.kappa/TWO_PI*1e3:.1f} MHz")
print(f"  Γ/2π  = {cfg_base.Gamma/TWO_PI*1e3:.0f} MHz")
print(f"  Δ/2π  = {(cfg_base.omega_d-cfg_base.omega0)/TWO_PI*1e3:.1f} MHz (base detuning)")
print(f"  ω_f/2π = {cfg_base.omega_f/TWO_PI:.3f} GHz (bath center)")

<a id="sec2"></a>
## 2  Rate Extraction from Both Methods

To compare Method A and Method C at a single $(\varepsilon, \Delta)$ point, we
extract the effective $|1\rangle \to |0\rangle$ decay rate from each:

- **Method A:** simulate $|1\rangle$ decay under Lindblad and fit the $P_{|1\rangle}(t)$
  decay to an exponential.
- **Method C:** compute the Floquet rate matrix $R$ and extract the dominant relaxation
  rate using the smallest non-zero eigenvalue of $R$.

The **fractional discrepancy** is then:

$$D(\varepsilon, \Delta) = \frac{|\Gamma_A - \Gamma_C|}{\Gamma_A}.$$

When $D < 10\%$ we say Method A is valid; when $D \sim 1$ Method A is completely wrong.

In [ ]:
def compute_rates(eps, delta, cfg_base):
    """
    Compute Method A and Method C effective decay rates.

    Parameters
    ----------
    eps : float
        Drive amplitude in rad/s.
    delta : float
        Drive detuning (ω_d - ω₀) in rad/s.
    cfg_base : DrivenKerrConfig
        Base configuration (epsilon and omega_d will be overridden).

    Returns
    -------
    Gamma_A : float
        Method A effective decay rate (rad/s).
    Gamma_C : float
        Method C effective decay rate (rad/s).
    """
    cfg = cfg_base.replace(
        epsilon=eps,
        omega_d=cfg_base.omega0 + delta,
    )

    # Time array: 5 decay times at reference kappa
    tlist = np.linspace(0, 5.0 / cfg_base.kappa, 80)
    rho0 = qt.fock_dm(cfg.N, 1)

    # Method A: Lindblad simulation → exponential fit
    res_A = run_lindblad(cfg, rho0, tlist)
    Gamma_A = effective_loss_rate(res_A, tlist, cfg)

    # Method C: Floquet rate matrix → dominant eigenvalue
    # Build initial population vector |1⟩ in Fock basis
    p0 = np.zeros(cfg.N)
    p0[1] = 1.0
    fm = run_full_floquet_markov(cfg, p0, tlist)
    Gamma_C = effective_decay_rate_from_R(fm["R"])

    return Gamma_A, Gamma_C


# Test at one representative point
eps_test  = TWO_PI * 10e-3   # 10 MHz
delta_test = TWO_PI * 5e-3   # 5 MHz detuning

print(f"Test point: ε/2π = {eps_test/TWO_PI*1e3:.1f} MHz, Δ/2π = {delta_test/TWO_PI*1e3:.1f} MHz")
Gamma_A_test, Gamma_C_test = compute_rates(eps_test, delta_test, cfg_base)

print(f"  Γ_A/2π = {Gamma_A_test/TWO_PI*1e3:.4f} MHz (Method A, Lindblad)")
print(f"  Γ_C/2π = {Gamma_C_test/TWO_PI*1e3:.4f} MHz (Method C, Floquet-Markov)")
if Gamma_A_test > 0 and Gamma_C_test > 0:
    disc = abs(Gamma_A_test - Gamma_C_test) / Gamma_A_test
    print(f"  Discrepancy |Γ_A - Γ_C|/Γ_A = {disc:.1%}")

<a id="sec3"></a>
## 3  2D Validity Map

We now sweep over a grid of $(\varepsilon, \Delta)$ values and compute the
fractional discrepancy at each point. The result is the **validity map**:

- **Green** (low discrepancy): Methods A and C agree — Lindblad is valid.
- **Red** (high discrepancy): Methods disagree — Lindblad significantly overestimates decay.
- **White (NaN)**: Floquet mode identification failed (level collision at strong drive
  + large detuning where $|0\rangle$ and $|1\rangle$ are equally mixed into two
  dressed Floquet states).

This is the main result of this notebook, reproducing the key figure from the
reference project (`sec6_error_budget.py`).

**Warning:** this sweep involves $n_{\rm eps} \times n_{\rm delta}$ Floquet mode
computations, each taking several seconds. With $N=8$, $n_{\rm eps}=12$,
$n_{\rm delta}=10$ expect $\sim 5{-}15$ minutes on a laptop.
Use BINDER_FAST parameters ($N=4$, $n_{\rm eps}=6$, $n_{\rm delta}=5$) for a
quick preview.

In [ ]:
# Define the 2D grid
# Epsilon: 0.5 MHz to 400 MHz (well into the strong-drive regime)
eps_arr   = np.geomspace(TWO_PI * 0.5e-3, TWO_PI * 0.4, n_eps)
# Detuning: ±50 MHz (symmetric around zero)
delta_arr = np.linspace(-TWO_PI * 0.05, TWO_PI * 0.05, n_delta)

print(f"2D validity map: {n_eps} × {n_delta} = {n_eps*n_delta} points")
print(f"  ε/2π: {eps_arr[0]/TWO_PI*1e3:.2f} to {eps_arr[-1]/TWO_PI*1e3:.0f} MHz (log-spaced)")
print(f"  Δ/2π: {delta_arr[0]/TWO_PI*1e3:.1f} to {delta_arr[-1]/TWO_PI*1e3:.1f} MHz")
print()

# map_2d[i, j] = discrepancy at (delta_arr[i], eps_arr[j])
map_2d = np.full((n_delta, n_eps), np.nan)

for i, delta in enumerate(delta_arr):
    for j, eps in enumerate(eps_arr):
        try:
            Gamma_A, Gamma_C = compute_rates(eps, delta, cfg_base)
            if Gamma_A > 0 and Gamma_C > 0 and np.isfinite(Gamma_A) and np.isfinite(Gamma_C):
                map_2d[i, j] = abs(Gamma_A - Gamma_C) / Gamma_A
        except Exception as exc:
            pass  # NaN for failed mode identification

    print(f"  Detuning row {i+1}/{n_delta} (Δ/2π = {delta/TWO_PI*1e3:.1f} MHz) done")

print(f"\nMap complete. Valid cells: {np.sum(np.isfinite(map_2d))}/{n_eps*n_delta}")
print(f"NaN cells (mode identification failure): {np.sum(np.isnan(map_2d))}")

In [ ]:
# Plot the validity map
eps_MHz   = eps_arr   / TWO_PI * 1e3
delta_MHz = delta_arr / TWO_PI * 1e3

fig, ax = plt.subplots(figsize=(8, 5))

# Mask NaN → displayed as white/grey by cmap.set_bad
disc_masked = np.ma.masked_invalid(map_2d)
cmap = plt.cm.RdYlGn_r.copy()
cmap.set_bad("lightgray")

im = ax.pcolormesh(
    eps_MHz, delta_MHz, disc_masked,
    cmap=cmap, vmin=0, vmax=1, shading="auto",
)

# 10 % validity contour
try:
    cs = ax.contour(
        eps_MHz, delta_MHz, map_2d,
        levels=[0.10],
        colors="white", linewidths=2.0,
    )
    ax.clabel(cs, fmt={0.10: "10%"}, fontsize=9, inline=True)
except Exception:
    pass  # contour may fail if map is too sparse

cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label(r"$|\Gamma_A - \Gamma_C| / \Gamma_A$", fontsize=10)

ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (MHz)", fontsize=11)
ax.set_ylabel(r"Drive detuning $\Delta/2\pi$ (MHz)", fontsize=11)
ax.set_title(
    "Relative discrepancy $|\\Gamma_A - \\Gamma_C| / \\Gamma_A$\n"
    "Green = agreement, Red = Methods A and C disagree, Gray = NaN (mode collision)",
    fontsize=10,
)
ax.axhline(0, color="white", ls=":", lw=0.8, alpha=0.5)
ax.grid(True, alpha=0.2, color="white", lw=0.5)

plt.tight_layout()
plt.show()

print("Map statistics:")
valid = map_2d[np.isfinite(map_2d)]
if len(valid) > 0:
    print(f"  Mean discrepancy (valid cells): {valid.mean():.1%}")
    print(f"  Cells with D < 10%: {np.sum(valid < 0.10)} / {len(valid)}")
    print(f"  Cells with D > 50%: {np.sum(valid > 0.50)} / {len(valid)}")

<a id="sec4"></a>
## 4  Physical Interpretation

**Reading the validity map:**

**Green region (agreement, $D < 10\%$):** Method A is valid.
This occupies the bottom-left corner — small $\varepsilon$ (weak drive) and small
$|\Delta|$ (near resonance). The sideband shift $\varepsilon^2/\Gamma$ is negligible
and the bath couples the bare $|1\rangle \to |0\rangle$ transition efficiently.

**Red region (disagreement, $D \sim 1$):** The dominant $k = -1$ Floquet sideband
has shifted far outside the bath Lorentzian ($\Gamma/2\pi = 100$ MHz wide),
reducing $J(\Delta_{m_0 m_1} + \omega_d)$ by a large factor. Method A, which
always uses $J(\omega_0)$, overestimates the decay rate by up to $20\times$.

**White/gray cells (NaN):** Floquet mode identification fails when a level
anti-crossing makes the two modes with highest overlap with $|0\rangle$ and
$|1\rangle$ indistinguishable. This occurs at strong drive + large detuning
where $|0\rangle$ and $|1\rangle$ are strongly mixed into two dressed states
with comparable overlap.

**Key message:** For any driven system with $\varepsilon/\Gamma \gtrsim 0.1$,
Method C (Floquet–Markov) should be used. Method A underestimates the
decoherence suppression by up to $20\times$ at $\varepsilon = K$.
This is not a numerical artefact — it is a genuine physical difference
between the dressed and undressed basis.

**Practical rule:** Before running any error budget, locate your operating
point $(\varepsilon, \Delta)$ on this map. If it lies in the green region,
Lindblad suffices. If it is in the red region, use the Floquet–Markov budget
from Module 3c instead.

*Lab note: the validity map is a one-time computation for a given device.
Once computed, it becomes a reference plot for all experiments on that device.
When designing a new gate, verify that the target drive amplitude lies in the
green region if Lindblad is to be trusted. If the gate requires strong drive
($\varepsilon \gtrsim \Gamma$), the Floquet budget is mandatory.*

<a id="sec5"></a>
## 5  Analytic Validity Boundary

The dominant mechanism for sideband frequency shift is that, at drive detuning
$\Delta = \omega_d - \omega_0$, the $k=-1$ Floquet sideband for the
$m_1 \to m_0$ transition is sampled at approximately:

$$\omega_{\rm sideband} \approx \omega_0 + \frac{\varepsilon^2}{\Gamma} + 2|\Delta|.$$

The bath coupling drops below $50\%$ of peak when $\omega_{\rm sideband}$ exceeds
$\omega_0 + \Gamma/2$ (the half-power point of the Lorentzian).
Setting $\delta\omega_{\rm sideband} = \Gamma/2$ and solving for $\varepsilon$:

$$\varepsilon_{\rm boundary} = \sqrt{\Gamma\left(\frac{\Gamma}{2} - 2|\Delta|\right)},
\qquad \text{for } |\Delta| < \Gamma/4.$$

For larger $|\Delta|$, even $\varepsilon = 0$ is outside the bath. The boundary
therefore vanishes at $|\Delta| = \Gamma/4 = 25$ MHz for our parameters.

In [ ]:
# Overlay the analytic boundary on the validity map
Gamma = cfg_base.Gamma

# Analytic boundary: epsilon_boundary(delta) for the 50%-coupling criterion
delta_boundary = np.linspace(-Gamma/4, Gamma/4, 200)  # rad/s
eps_boundary = np.sqrt(Gamma * (Gamma/2 - 2*np.abs(delta_boundary)))

# Plot map again with boundary overlay
fig, ax = plt.subplots(figsize=(8, 5))
disc_masked = np.ma.masked_invalid(map_2d)
cmap = plt.cm.RdYlGn_r.copy()
cmap.set_bad("lightgray")
im = ax.pcolormesh(
    eps_MHz, delta_MHz, disc_masked,
    cmap=cmap, vmin=0, vmax=1, shading="auto",
)

# 10% contour from data
try:
    cs = ax.contour(eps_MHz, delta_MHz, map_2d, levels=[0.10],
                    colors="white", linewidths=2.0)
    ax.clabel(cs, fmt={0.10: "10%"}, fontsize=9, inline=True)
except Exception:
    pass

# Analytic boundary (50% bath coupling criterion)
ax.plot(
    eps_boundary / TWO_PI * 1e3,
    delta_boundary / TWO_PI * 1e3,
    "y--", lw=2, label=r"Analytic boundary ($\delta\omega_{\rm sb} = \Gamma/2$)",
    zorder=5,
)

cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label(r"$|\Gamma_A - \Gamma_C| / \Gamma_A$", fontsize=10)
ax.set_xlabel(r"Drive amplitude $\varepsilon/2\pi$ (MHz)", fontsize=11)
ax.set_ylabel(r"Drive detuning $\Delta/2\pi$ (MHz)", fontsize=11)
ax.set_title(
    "Validity map with analytic boundary (dashed yellow = 50% bath coupling)",
    fontsize=10,
)
ax.legend(loc="upper right", fontsize=9)
ax.axhline(0, color="white", ls=":", lw=0.8, alpha=0.5)
ax.grid(True, alpha=0.2, color="white", lw=0.5)
plt.tight_layout()
plt.show()

# Verify NaN cells cluster at strong drive + large detuning
nan_mask = np.isnan(map_2d)
if nan_mask.any():
    nan_eps_idx, nan_del_idx = np.where(nan_mask.T)  # note: map shape is (n_delta, n_eps)
    # Correct: map_2d[i, j] -> i=delta, j=eps
    nan_i, nan_j = np.where(nan_mask)
    print(f"NaN cells: {nan_mask.sum()} total")
    print(f"  Mean ε/2π of NaN cells: {eps_arr[nan_j].mean()/TWO_PI*1e3:.1f} MHz")
    print(f"  Mean |Δ|/2π of NaN cells: {np.abs(delta_arr[nan_i]).mean()/TWO_PI*1e3:.1f} MHz")
    print("  (NaN cells are concentrated at strong drive + large detuning — as expected)")
else:
    print("No NaN cells in this map (all points converged successfully).")

# Analytic boundary at Δ=0 (max allowed ε)
eps_at_zero_delta = np.sqrt(Gamma * Gamma / 2)
print(f"\nAnalytic ε_boundary at Δ=0: {eps_at_zero_delta/TWO_PI*1e3:.1f} MHz")
print(f"For reference: K/2π = {cfg_base.K/TWO_PI*1e3:.0f} MHz")

<a id="sec6"></a>
## 6  Exercises

### Exercise 1: Wide bath validity map

Repeat the 2D validity map with $\Gamma/2\pi = 500$ MHz (wider bath).
Does the green region expand? By how much does the 10% boundary shift?

**Expected:** the green region should expand significantly, because the wider
bath is flatter over the range of sideband shifts produced by the same drive
amplitudes. The analytic boundary shifts from $\sim 70$ MHz to $\sim 350$ MHz
at $\Delta = 0$.

In [ ]:
# YOUR CODE HERE
# Hint:
# cfg_wide = cfg_base.replace(Gamma=TWO_PI * 500e-3)
# map_wide = np.full((n_delta, n_eps), np.nan)
# for i, delta in enumerate(delta_arr):
#     for j, eps in enumerate(eps_arr):
#         try:
#             Ga, Gc = compute_rates(eps, delta, cfg_wide)
#             if Ga > 0 and Gc > 0:
#                 map_wide[i, j] = abs(Ga - Gc) / Ga
#         except Exception:
#             pass
# Then plot map_wide with the same color scale

### Exercise 2: Maximum valid drive amplitude

From the validity map, estimate the maximum $\varepsilon/2\pi$ at which Lindblad
is valid (10% criterion) for our default parameters at zero detuning ($\Delta = 0$).

Compare your answer to the analytic estimate $\varepsilon_{\rm boundary}(\Delta=0)
= \Gamma/\sqrt{2}$. Do they agree? If not, why might the map give a different
answer than the simple analytic formula?

In [ ]:
# YOUR CODE HERE
# Hint: find the row closest to delta=0 in the map,
# then find the column where map_2d[row, :] first exceeds 0.10.
# Use np.searchsorted or np.interp on that row.
#
# row_zero_delta = np.argmin(np.abs(delta_arr))
# disc_row = map_2d[row_zero_delta, :]
# valid_mask = np.isfinite(disc_row) & (disc_row < 0.10)
# max_valid_eps = eps_arr[valid_mask].max() if valid_mask.any() else 0.0
# print(f"Max valid ε at Δ=0: {max_valid_eps/TWO_PI*1e3:.1f} MHz")